# Spotify Library — Data Cleaning & Preprocessing

Converts `spotifyLibrary.csv` into a ML-ready dataset for K-Prototypes clustering.

Every decision here is grounded in the findings from **Spotify EDA - Comprehensive.ipynb**.

**Pipeline**
1. Load raw data & audit quality
2. Drop meta / identifier columns
3. Engineer `duration_min`, drop `duration_ms`
4. Remove duplicate rows
5. Drop low-signal features (`loudness`, `time_signature`)
6. Apply `log1p` transforms to right-skewed continuous features
7. `MinMaxScaler` on all continuous features
8. Cast categorical columns to `str` for K-Prototypes
9. Validate & save `spotifyLibrary_cleaned.csv`

---
## 1. Setup & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})
COLOR_MAIN = '#1DB954'
COLOR_ACC  = '#191414'

RAW_PATH  = 'spotifyLibrary.csv'
OUT_PATH  = 'spotifyLibrary_cleaned.csv'

df_raw = pd.read_csv(RAW_PATH, index_col=0)
print(f'Raw shape: {df_raw.shape}')
df_raw.head(3)

---
## 2. Baseline Quality Audit

Confirm the EDA findings before any transformations: no nulls, 8 duplicate rows, correct dtypes.

In [ ]:
print('=== Missing Values ===')
print(df_raw.isnull().sum())
print(f'\nTotal nulls : {df_raw.isnull().sum().sum()}')
print(f'Duplicates  : {df_raw.duplicated().sum()}')
print(f'\nDtypes:')
print(df_raw.dtypes)

---
## 3. Drop Identifier / Meta Columns

These columns are API bookkeeping — they carry no audio signal and would corrupt distance calculations.

In [ ]:
META_COLS = ['type', 'id', 'uri', 'track_href', 'analysis_url']

df = df_raw.drop(columns=[c for c in META_COLS if c in df_raw.columns])
print(f'Columns after dropping meta: {df.columns.tolist()}')

---
## 4. Engineer `duration_min` & Drop `duration_ms`

`duration_ms` is in the tens of thousands — converting to minutes puts it on a human-interpretable scale and prevents the large raw magnitude from dominating distance metrics before scaling.

In [ ]:
df['duration_min'] = df['duration_ms'] / 60_000
df = df.drop(columns=['duration_ms'])

print(f'duration_min  min={df["duration_min"].min():.2f}  '
      f'max={df["duration_min"].max():.2f}  '
      f'mean={df["duration_min"].mean():.2f}')

---
## 5. Remove Duplicate Rows

EDA found 8 exact duplicates. Keeping them would inflate the apparent density of certain audio profiles and bias cluster centroids.

In [ ]:
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_after  = len(df)

print(f'Rows before : {n_before}')
print(f'Rows removed: {n_before - n_after}')
print(f'Rows after  : {n_after}')

---
## 6. Drop Low-Signal Features

### 6a. `loudness` — redundant with `energy`
EDA showed energy ↔ loudness Pearson r ≈ 0.70. In K-Prototypes (and any distance-based method) keeping both dimensions effectively double-counts the same underlying signal. `energy` is retained because it is already normalised to [0, 1], whereas `loudness` is in negative dB and harder to interpret.

### 6b. `time_signature` — near-zero variance
~95% of tracks are 4/4. A feature that is nearly constant across the dataset contributes almost nothing to cluster separation and can skew K-Prototypes' categorical dissimilarity term.

In [ ]:
print('time_signature value counts:')
print(df['time_signature'].value_counts())
print(f'\n4/4 proportion: {(df["time_signature"] == 4).mean():.1%}')

In [ ]:
DROP_FEATURES = ['loudness', 'time_signature']
df = df.drop(columns=DROP_FEATURES)

print(f'Remaining columns ({df.shape[1]}): {df.columns.tolist()}')

---
## 7. `log1p` Transform Skewed Continuous Features

EDA identified four heavily right-skewed features (|skew| > 0.75). K-Prototypes uses Euclidean distance for continuous dimensions — a spike-shaped distribution inflates distances in that dimension and lets a handful of extreme values dominate cluster assignment.

`log1p(x)` = `log(1 + x)` compresses the long right tail while handling exact zeros safely (unlike `log(x)`).

| Feature | Raw skew | Post-transform skew (approx) |
|---|---|---|
| `instrumentalness` | > 3 | near 0 |
| `speechiness` | > 2 | < 1 |
| `liveness` | > 1 | < 0.5 |
| `duration_min` | > 1 | < 0.5 |

In [ ]:
LOG_COLS = ['instrumentalness', 'speechiness', 'liveness', 'duration_min']

skew_before = {col: round(skew(df[col]), 3) for col in LOG_COLS}

for col in LOG_COLS:
    df[col] = np.log1p(df[col])

skew_after = {col: round(skew(df[col]), 3) for col in LOG_COLS}

skew_report = pd.DataFrame({'skew_before': skew_before, 'skew_after': skew_after}).T
print('Skewness before and after log1p:')
skew_report

In [ ]:
# Visual confirmation — before/after distributions for each transformed feature
# Reload originals for the 'before' panel
df_pre = pd.read_csv(RAW_PATH, index_col=0).drop_duplicates().reset_index(drop=True)
df_pre['duration_min'] = df_pre['duration_ms'] / 60_000

fig, axes = plt.subplots(len(LOG_COLS), 2, figsize=(12, 3 * len(LOG_COLS)),
                          constrained_layout=True)

for row, col in enumerate(LOG_COLS):
    raw   = df_pre[col].dropna()
    trans = df[col].dropna()

    for ax, data, label in zip(
        axes[row],
        [raw, trans],
        [f'{col}  (raw, skew={skew(raw):.2f})',
         f'{col}  (log1p, skew={skew(trans):.2f})']):

        ax.hist(data, bins=40, color=COLOR_MAIN, alpha=0.55,
                edgecolor='white', linewidth=0.4, density=True)
        data.plot.kde(ax=ax, color=COLOR_ACC, linewidth=2)
        ax.set_title(label, fontsize=10)
        ax.set_ylabel('Density')

fig.suptitle('log1p Transform — Before vs After', fontsize=14, fontweight='bold')
plt.show()

---
## 8. MinMax Scale Continuous Features

All continuous features are scaled to [0, 1] so no single feature dominates K-Prototypes' Euclidean distance term due to a wider numeric range (e.g. `tempo` is in the hundreds while `danceability` is 0–1).

**Categorical columns (`key`, `mode`) are deliberately excluded** — they will be handled by K-Prototypes' categorical dissimilarity term and must remain as discrete labels.

In [ ]:
CAT_COLS  = ['key', 'mode']
CONT_COLS = [c for c in df.columns if c not in CAT_COLS]

print(f'Continuous features to scale ({len(CONT_COLS)}): {CONT_COLS}')
print(f'Categorical features kept as-is ({len(CAT_COLS)}): {CAT_COLS}')

In [ ]:
scaler = MinMaxScaler()
df[CONT_COLS] = scaler.fit_transform(df[CONT_COLS])

print('Continuous feature ranges after scaling:')
df[CONT_COLS].agg(['min', 'max']).T

---
## 9. Cast Categorical Columns to String

K-Prototypes expects categorical columns to be non-numeric so it routes them through the Hamming-style dissimilarity term instead of Euclidean distance. Casting to `str` satisfies this requirement without creating ordinal assumptions.

In [ ]:
for col in CAT_COLS:
    df[col] = df[col].astype(str)

print('Dtypes after casting:')
print(df.dtypes)

---
## 10. Validation

Final checks before saving: no nulls, correct shape, all continuous features in [0, 1], categorical columns are strings.

In [ ]:
print('=== Final Dataset Shape ===')
print(f'{df.shape[0]:,} tracks  ×  {df.shape[1]} features')
print()

print('=== Null Check ===')
null_total = df.isnull().sum().sum()
print(f'Total nulls: {null_total}', '✓' if null_total == 0 else '✗ FIX REQUIRED')
print()

print('=== Continuous Feature Range Check (all should be [0.0, 1.0]) ===')
range_check = df[CONT_COLS].agg(['min', 'max']).T
range_check['in_range'] = (range_check['min'] >= 0) & (range_check['max'] <= 1)
print(range_check.to_string())
print()

print('=== Categorical Column Dtypes ===')
for col in CAT_COLS:
    print(f'  {col}: dtype={df[col].dtype}  unique={df[col].nunique()} values')

In [ ]:
# Distribution overview of the cleaned, scaled continuous features
fig, axes = plt.subplots(2, 4, figsize=(18, 8), constrained_layout=True)
axes = axes.flatten()

for i, col in enumerate(CONT_COLS):
    ax = axes[i]
    data = df[col]
    ax.hist(data, bins=40, color=COLOR_MAIN, alpha=0.55,
            edgecolor='white', linewidth=0.4, density=True)
    data.plot.kde(ax=ax, color=COLOR_ACC, linewidth=2)
    ax.axvline(data.mean(), color='#e74c3c', linestyle='--',
               linewidth=1.4, label=f'mean={data.mean():.2f}')
    ax.set_title(col)
    ax.set_xlim(0, 1)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Cleaned & Scaled Feature Distributions', fontsize=15, fontweight='bold')
plt.show()

In [ ]:
# Correlation heatmap on cleaned data — confirm energy/acousticness split still intact
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[CONT_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
            linecolor='white', ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Cleaned Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Save Cleaned Dataset

In [ ]:
df.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

---
## 12. Cleaning Summary

| Step | Action | Rationale |
|---|---|---|
| Meta drop | Removed `type`, `id`, `uri`, `track_href`, `analysis_url` | No audio signal; would corrupt distance metrics |
| Feature engineer | `duration_ms` → `duration_min` | Human-interpretable; same relative scale |
| Deduplication | Removed 8 exact duplicate rows | Prevents centroid bias |
| Feature drop | Removed `loudness` | Energy ↔ Loudness r ≈ 0.70; energy retained as already-bounded [0,1] |
| Feature drop | Removed `time_signature` | ~95% 4/4; near-zero variance adds noise, not signal |
| log1p transform | `instrumentalness`, `speechiness`, `liveness`, `duration_min` | Corrects heavy right skew before distance-based clustering |
| MinMax scale | All 8 continuous features → [0, 1] | Prevents wide-range features from dominating Euclidean distance |
| Categorical cast | `key`, `mode` → `str` | Satisfies K-Prototypes' categorical dissimilarity routing |

**Output:** `spotifyLibrary_cleaned.csv` — 8 scaled continuous features + 2 categorical features, ready for K-Prototypes with `n_clusters` sweep 5–9.